# Ditaso (2026) - Data processing: stocks

This notebook mines and processes the stock data needed. Read the [README.md](README.md) file for context about the project.

## Data and methodology

### Raw data

**Sources:** Intraday tick data from the [NYSE TAQ database](https://www.nyse.com/data-products/catalog/daily-taq).

**Format:**

The data is can be easily stored as an SQLite3 database with the main table `trades` of the following schema:

```sql
CREATE TABLE trades (
    permno    INTEGER NOT NULL, -- Permanent number of the stock
    ts_ny     INTEGER NOT NULL, -- Timestamp in New York time (epoch seconds)
    price     REAL    NOT NULL, -- Price of the trade
    size      INTEGER NOT NULL, -- Size of the trade
    ex        TEXT    NOT NULL, -- Exchange code
    corr      TEXT    NOT NULL, -- Correction code
    ticker    TEXT    NOT NULL, -- Stock ticker symbol
    ambiguous INTEGER NOT NULL, -- Flag if the trade is ambiguous (0 or 1)
    PRIMARY KEY (permno, ts_ny)
) WITHOUT ROWID
```

Further documentation might be found looking for the relevant websites.

For the confection of this notebook, the data was given in a private Dropbox folder. The [TODO]() file contains a script to download it into the [data/trades.db](data/trades.db) file. 

**Qualities:**

- The data should be ordered by `permno` then `ts_ny`
- The `price` should never be negative or zero
- `ex`, `corr`, and `ticker` should have only the correct levels.
- There should be data only for trading days.

All are guaranteed by the data source, and are checked informally.


### Methodology

First, some initial filters are applied (@sec-stocks-filter). Then, the data is transformed from a `permno` ~ `second` ~ `price` data into a `permno` ~ `minute` ~ `log return` data (@sec-stocks-agg).

The calculations are done via [DuckDB](https://duckdb.org/) SQL queries.

## Code setup

Codes for debugging and running SQL inside notebooks:

In [1]:
%reset -f
import importlib

import duckdb
from IPython.core.magic import register_cell_magic

sql_queries: dict[str, str] = {}

@register_cell_magic
def register_query(line: str, cell: str) -> None:
    query_name = line.strip()
    if not query_name:
        raise ValueError("You must provide a name for the query.")
    sql_queries[query_name] = cell
    return None

Setting up DuckDB infrastructure:

In [2]:
%%register_query duckdb_infra
INSTALL sqlite;
LOAD sqlite;
ATTACH 'data/trades.db' AS db_trades (TYPE SQLITE);

SET enable_progress_bar = true;
SET enable_progress_bar_print = true;

PRAGMA threads = 6;
PRAGMA memory_limit = '14GB';

In [3]:
#| output: false
duckdb.execute(sql_queries["duckdb_infra"])

Core modules:

In [4]:
import os
if os.path.basename(os.getcwd()) == "src": os.chdir("..")

import math

import polars as pl
import numpy as np

import plotnine as gg
from plotnine import ggplot, aes

import src.utils as ut
importlib.reload(ut)
from src.parameters import PARAMETERS as PARS

## Initial data filtering {#sec-stocks-filter}

An initial filter is applied before the processing logic, to remove uncessary data and introduce a better file format (`.parquet`).

**Filters:**

- Exchanges: NYSE, NYSE-Arca, NASDAQ, and AMEX (code shares 10 and 11 at the CRSP database) (`ex IN ('P', 'N', 'Q', 'A')`). 
- Types of trades: regular trades (`corr = '00'`).
- Sample period(same as factors):
    - Range: 2001-01-02 to 2017-12-29. Stocks that are listed or delisted during this period are kept.
    - Hours: regular trading hours (09:30:00 - 15:59:59 ET).
    - Days: will be applied only at the day-level data.

This will be implemented via the following query:

In [5]:
%%register_query raw__to_filtered_parquet

-- ?1: start epoch seconds (PARS.date_start)
-- ?2: end epoch seconds (PARS.date_end)

COPY (
	SELECT
		t.permno,
		t.ts_ny,
		t.price,
		t.ticker
	FROM db_trades.trades t
	WHERE
		t.ts_ny BETWEEN ?1 AND ?2
		AND t.corr = '00'
		AND t.ex IN ('P', 'N', 'Q', 'A')
		AND (t.ts_ny % 86400) BETWEEN 34200 AND 57599 -- 09:30:00 - 15:59:59
)
TO 'data/trades_filtered.parquet' (
	FORMAT PARQUET,
	COMPRESSION 'ZSTD',
	ROW_GROUP_SIZE 100000
);

In [6]:
if False: # Only excecute once
    duckdb.execute(
        sql_queries["trade_returns_aggregation"],
        [
            int(x.astype("datetime64[s]").astype(np.int64))
            for x in (PARS.date_start, PARS.date_end)
        ]
    )

For utility, save the number of observations for each `permno` and each (`permno`, minute) pair:

In [7]:
%%register_query permno_counts
COPY (
    SELECT 
        permno, 
        COUNT(*) as obs_count
    FROM 'data/trades_filtered.parquet'
    GROUP BY permno
    ORDER BY obs_count DESC
) TO 'data/permno_counts.csv' (HEADER, DELIMITER ',');

In [8]:
if False: # Only excecute once
    duckdb.execute(query = sql_queries["permno_counts"])

data_permno = pl.read_csv("data/permno_counts.csv")

In [9]:
%%register_query day_counts
COPY (
    SELECT 
        permno,
        (ts_ny // (60 * 60 * 24)) as d,
        COUNT(*) as obs_count
    FROM 'data/trades_filtered.parquet'
    GROUP BY permno, d
) TO 'data/day_counts.csv' (HEADER, DELIMITER ',');

In [10]:
if False: # Only excecute once
    duckdb.execute(query = sql_queries["day_counts"])

With the minute counts, we can check if the data is ordered and conforms with trading days. It is expected that there are data for non-full trading days (days with early stoppages, etc.), these are defined via the factor data (see the factors notebook/section).

In [11]:
data_days = pl.read_csv(
    "data/day_counts.csv"
).with_columns(
    pl.from_epoch(pl.col("d"), time_unit = "d")
)

days_unique = data_days["d"].to_numpy().astype("datetime64[D]")

if not np.isin(days_unique, PARS.trading_days).all():
    raise ValueError("There are days in the data that are not trading days.")

if not np.isin(days_unique, PARS.trading_days_full).all():
    print("There are days in the data that are not full trading days.")

There are days in the data that are not full trading days.


In [12]:
g_data_days = (
    data_days.with_columns(pl.col("obs_count") / PARS.blocks_1min)
    #.filter(np.isin(data_days["d"], PARS.trading_days_full))
    .group_by("d")
    .agg(
        pl.median("obs_count").alias("median"),
        *[
            pl.quantile("obs_count", q / 10).alias(f"q{q}")
            for q in [0.4, 0.6, 0.3, 0.7, 0.2, 0.8, 0.1, 0.9]
        ]
    )
)

g_fill_colors = {
    "10% - 90%": "#ffb399",
    "20% - 80%": "#ff6699",
    "30% - 70%": "#ff0000",
    "40% - 60%": "#ffffff",
    "median": "black"
}

(
    ggplot(g_data_days, aes(x = "d", group = 1)) +
    gg.geom_ribbon(aes(ymin = "q1", ymax = "q9", fill = "'10% - 90%'")) +
    gg.geom_ribbon(aes(ymin = "q2", ymax = "q8", fill = "'20% - 80%'")) +
    gg.geom_ribbon(aes(ymin = "q3", ymax = "q7", fill = "'30% - 70%'")) +
    gg.geom_ribbon(aes(ymin = "q4", ymax = "q6", fill = "'40% - 60%'")) +
    gg.geom_line(aes(y = "median", color = "'median'"), size = 0.25) +
    gg.scale_x_date(date_labels = "%Y", date_breaks = "1 year") +
    gg.scale_fill_manual(values = g_fill_colors) +
    gg.scale_color_manual(values = g_fill_colors) +
    gg.scale_y_continuous(labels = lambda bs: ["{:.0%}".format(b) for b in bs]) +
    gg.labs(
        title = "Proportion of minutes with data within each day",
        subtitle = (
            f"Quantiles calculated each day across all {data_permno.shape[0]}"
            " stocks"
        ),
        y = "Proportion of minutes with data", x = "Day",
        fill = "Quantiles", color = " "
    ) +
    gg.theme_bw() +
    gg.theme(
        axis_text_x = gg.element_text(rotation = 45, hjust = 1),
        figure_size = (8, 5)
    )
)

PlotnineError: "Could not evaluate the 'ymin' mapping: 'q1' (original error: name 'q1' is not defined)"

In [ ]:
days_low_obs = g_data_days.filter(
    pl.col("median").diff() <= -0.4
)["d"].to_numpy()

print(ut.format_days_byyear(days_low_obs))


- 2001: 08-06, 04-24, 08-30, 12-24, 05-18, 01-04, 08-01, 03-20, 08-16, 08-13, 04-25, 10-09, 01-29, 06-15, 01-23, 01-17, 07-03, 11-23, 07-09, 08-14, 08-08, 04-23, 08-02, 04-20, 01-09, 01-03, 05-14, 01-30, 02-05, 02-02, 03-13, 07-16
- 2002: 12-24
- 2003: 12-24, 11-28, 12-26
- 2004: 11-26
- 2005: 11-25
- 2007: 12-24
- 2009: 12-24
- 2011: 11-25
- 2012: 07-03, 11-23, 12-24
- 2013: 11-29
- 2014: 12-26


In [ ]:
(
    data_days
    .filter(np.isin(data_days["d"], days_low_obs))
    .group_by("d")
    .agg(pl.col("obs_count").median())
)

d,obs_count
date,f64
2005-08-04,279.0
2014-03-17,276.0
2009-03-03,361.0
2016-02-18,308.0
2006-02-08,303.5
…,…
2007-11-12,367.0
2008-12-15,343.0
2008-10-02,351.0


## Aggregation of 1-minute log returns {#sec-stocks-agg}

To aggregate the data, for each `permno`:

- For each 1-minute block $m$:
    - Get the last seen price before the end of $m$'s block, $p_m$.
    - Minimal liquidity filter: at most 2.5% of the trading days (excluding pre-listing and post-delisting days) have less than 80% of blocks with no trades.
    - Calculate $r^1_m = \ln(p_t / p_{m-1})$.
    - Exclude $p_{\text{9:30}}$ to remove overnight effects.
    - Exclude non-full trading days. Defined by the full days of the factor data (see [src/data_factors.ipynb](src/data_factors.ipynb)).
- Save a [data/stocks_1min_returns.parquet](data/stocks_1min_returns.parquet) file with the columns `permno`, `ts_ny`, `log-return`, and `ticker`.

Note that the "last seen price" is not the undisputed correct price to use for calculating returns. Other methods include the midpoint of the bid-ask spread and the volume-weighted average price.

The data is larger-than-memory, so it is processed in chunks of `permno` values. The chunks are definde below, striving for a balance in the number of rows in each chunk. Each chunk has 5% of the rows, a number defined by the memory constrains (16GB RAM) of the machine used to process the data.

In [25]:
data_counts = pl.read_csv("data/permno_counts.csv")

chunk_fraction = 0.05
chunk_cutoff = (
    math.floor(data_counts["obs_count"].sum() * chunk_fraction)
    * 0.9 # helps avoid last chunk being too small
)

chunks = []
current_stock = 0

for _ in range(math.ceil(1 / chunk_fraction)):
    c = []
    c_rows = 0
    while c_rows < chunk_cutoff and current_stock < len(data_counts):
        c.append(data_counts["permno"][current_stock])
        c_rows += data_counts["obs_count"][current_stock]
        current_stock += 1
    chunks.append(c)

The transformation could be done with DuckDB, but it is done with Polars, also multithreaded and written in Rust. DuckDB consumes less memory, could use less chunks, but I found Polars to be faster overall.

Note that we could save the information of how many observations were within each minute block, but that information is not needed at the moment.

In [26]:
lazy_data_trades = pl.scan_parquet("data/trades_filtered.parquet")

def lazy_pipeline_agg(chunk: list[int]) -> pl.LazyFrame:
    trading_days_extra = PARS.trading_days[
        ~ np.isin(PARS.trading_days, PARS.trading_days_full)
    ].astype("int64")

    pipeline = (
        lazy_data_trades
        .filter(pl.col("permno").is_in(chunk)) # Debug
        .with_columns(
            (pl.col("ts_ny") // 60).alias("m")
        )
        .group_by(["permno", "m"])
        .agg([
            pl.col("price").last().alias("price"), # Assuming sorted data
            pl.col("ticker").last().alias("ticker")
        ])
        .sort("m") # Groupby scrambles order, reorder for lag
        .with_columns(
            (pl.col("price") / pl.col("price").shift(1).over("permno"))
            .log()
            .alias("log_return")
        )
        .filter(
            (pl.col("m") % 1440 != 570)  # Exclude 9:30 AM
            & (pl.col("log_return").is_not_null()) # Excludes null lag
            & (~ pl.col("m").is_in(trading_days_extra)) # Exclude extra trading days
        )
        .select(["m", "permno", "log_return", "ticker"])
    )
    return pipeline
# theoretically, we dont need to keep the groupby, no need for sort, and create the lag col (within groupby)

For each chunk, the pipeline is executed and the results are saved in a parquet file such as [data/stocks_1min_returns/chunk_1.parquet](data/stocks_1min_returns/chunk_1.parquet):

In [12]:
for i, chunk in enumerate(chunks):
    print(f"Processing chunk {i+1}/{len(chunks)} with {len(chunk)} permnos")
    lazy_pipeline_agg(chunk).sink_parquet(
        f"data/stocks_1min_returns/chunk_{i}.parquet",
        compression = "zstd",
        engine = "streaming",
        row_group_size = 500_000
    )

Processing chunk 1/20 with 45 permnos
Processing chunk 2/20 with 46 permnos
Processing chunk 3/20 with 47 permnos
Processing chunk 4/20 with 49 permnos
Processing chunk 5/20 with 51 permnos
Processing chunk 6/20 with 53 permnos
Processing chunk 7/20 with 55 permnos
Processing chunk 8/20 with 58 permnos
Processing chunk 9/20 with 60 permnos
Processing chunk 10/20 with 64 permnos
Processing chunk 11/20 with 68 permnos
Processing chunk 12/20 with 72 permnos
Processing chunk 13/20 with 77 permnos
Processing chunk 14/20 with 84 permnos
Processing chunk 15/20 with 91 permnos
Processing chunk 16/20 with 100 permnos
Processing chunk 17/20 with 112 permnos
Processing chunk 18/20 with 128 permnos
Processing chunk 19/20 with 150 permnos
Processing chunk 20/20 with 178 permnos
